In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: C:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP


In [2]:
from scripts.query_parser import QueryParser

parser = QueryParser()

In [3]:
query = "3 bed under 700k"

filters = parser.parse(query)

print("Query:", query)
print("Filters:", filters)

Query: 3 bed under 700k
Filters: {'price_max': 700000, 'bedrooms': 3}


In [4]:
from pathlib import Path

schema_path = project_root / "data" / "schema.json"

print("Schema path:", schema_path)
print("File exists:", schema_path.exists())

Schema path: C:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\data\schema.json
File exists: True


In [5]:

from scripts.schema_validator import SchemaValidator

validator = SchemaValidator(str(schema_path))

print("SchemaValidator loaded successfully!")

SchemaValidator loaded successfully!


In [7]:
sql, params = parser.to_sql(filters, validator)

print("SQL:", sql)
print("Parameters:", params)

SQL: SELECT * FROM rets_property WHERE L_SystemPrice <= %s AND L_Keyword2 = %s
Parameters: (700000, 3)


In [8]:

queries = [
    "3 bed under 700k",
    "4 bedrooms below 900k",
    "2+ beds under 500k",
    "5 br max 1.2m",
    "3 bedrooms in Irvine",
    "2 bed in Beverly Hills under 2m",
    "homes between 500k and 800k",
    "at least 3 bedrooms",
    "no more than 2 bathrooms",
    "houses with a pool",
    "homes without a garage",
    "4 bedrooms under 1m in Lathrop"
]

results = []

for query in queries:
    filters = parser.parse(query)
    valid, errors = validator.validate_query(filters)

    results.append({
        "Query": query,
        "Filters": filters,
        "Valid": valid,
        "Errors": errors
    })

import pandas as pd

df = pd.DataFrame(results)
display(df)


,Query,Filters,Valid,Errors
0,3 bed under 700k,"{'price_max': 700000, 'bedrooms': 3}",True,[]
1,4 bedrooms below 900k,"{'price_max': 900000, 'bedrooms': 4}",True,[]
2,2+ beds under 500k,"{'price_max': 500000, 'bedrooms_min': 2}",True,[]
3,5 br max 1.2m,{'bedrooms': 5},True,[]
4,3 bedrooms in Irvine,"{'bedrooms': 3, 'city': 'Irvine'}",True,[]
5,2 bed in Beverly Hills under 2m,"{'price_max': 2000000, 'bedrooms': 2, 'city': ...",False,[City 'Beverly Hills' not found in database ci...
6,homes between 500k and 800k,"{'price_min': 500000, 'price_max': 800000}",True,[]
7,at least 3 bedrooms,{'bedrooms_min': 3},True,[]
8,no more than 2 bathrooms,{'bathrooms_max': 2},True,[]
9,houses with a pool,{'amenities': ['pool']},True,[]


In [9]:
query = "4 bedrooms under 1m in Lathrop"

filters = parser.parse(query)
valid, errors = validator.validate_query(filters)

print("Filters:", filters)
print("Valid:", valid)

if valid:
    sql, params = parser.to_sql(filters, validator)
    print("SQL:", sql)
    print("Parameters:", params)
else:
    print("Errors:", errors)

Filters: {'price_max': 1000000, 'bedrooms': 4, 'city': 'Lathrop'}
Valid: False
Errors: ["City 'Lathrop' not found in database city list"]


In [10]:
print(validator.schema["valid_cities"])

['Irvine', 'Portland', 'Los Angeles', 'San Diego', 'San Francisco', 'Anaheim', 'Newport Beach', 'Costa Mesa', 'Santa Ana', 'Tustin', 'Long Beach', 'Pasadena', 'Sacramento', 'Riverside', 'Oakland', 'San Jose', 'Santa Monica']


In [11]:
import json

schema_path = project_root / "data" / "schema.json"

with open(schema_path, "r", encoding="utf-8") as f:
    schema = json.load(f)

if "Lathrop" not in schema["valid_cities"]:
    schema["valid_cities"].append("Lathrop")

with open(schema_path, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=4)

print("Lathrop added!")

Lathrop added!


In [12]:
from scripts.schema_validator import SchemaValidator

validator = SchemaValidator(str(schema_path))

In [13]:
query = "4 bedrooms under 1m in Lathrop"

filters = parser.parse(query)
valid, errors = validator.validate_query(filters)

print("Filters:", filters)
print("Valid:", valid)
print("Errors:", errors)

if valid:
    sql, params = parser.to_sql(filters, validator)
    print("SQL:", sql)
    print("Parameters:", params)

Filters: {'price_max': 1000000, 'bedrooms': 4, 'city': 'Lathrop'}
Valid: True
Errors: []
SQL: SELECT * FROM rets_property WHERE L_SystemPrice <= %s AND L_Keyword2 = %s AND L_City = %s
Parameters: (1000000, 4, 'Lathrop')


In [14]:
gold_queries = [
    {
        "query": "3 bed under 700k",
        "expected": {"price_max": 700000, "bedrooms": 3}
    },
    {
        "query": "4 bedrooms below 900k",
        "expected": {"price_max": 900000, "bedrooms": 4}
    },
    {
        "query": "2+ beds under 500k",
        "expected": {"price_max": 500000, "bedrooms_min": 2}
    },
    {
        "query": "homes between 400k and 700k",
        "expected": {"price_min": 400000, "price_max": 700000}
    },
    {
        "query": "3 bedrooms in Irvine",
        "expected": {"bedrooms": 3, "city": "Irvine"}
    }
]

In [15]:

import pandas as pd

results = []

for example in gold_queries:
    query = example["query"]
    expected = example["expected"]

    predicted = parser.parse(query)

    # Exact match: every filter must match
    correct = predicted == expected

    results.append({
        "Query": query,
        "Expected": expected,
        "Predicted": predicted,
        "Correct": correct
    })

df = pd.DataFrame(results)

accuracy = df["Correct"].mean() * 100

display(df)

print(f"Total queries: {len(df)}")
print(f"Correct queries: {df['Correct'].sum()}")
print(f"Parsing accuracy: {accuracy:.2f}%")

if accuracy >= 90:
    print("PASS: Accuracy meets the 90% requirement!")
else:
    print("BELOW TARGET: Accuracy is less than 90%.")


,Query,Expected,Predicted,Correct
0,3 bed under 700k,"{'price_max': 700000, 'bedrooms': 3}","{'price_max': 700000, 'bedrooms': 3}",True
1,4 bedrooms below 900k,"{'price_max': 900000, 'bedrooms': 4}","{'price_max': 900000, 'bedrooms': 4}",True
2,2+ beds under 500k,"{'price_max': 500000, 'bedrooms_min': 2}","{'price_max': 500000, 'bedrooms_min': 2}",True
3,homes between 400k and 700k,"{'price_min': 400000, 'price_max': 700000}","{'price_min': 400000, 'price_max': 700000}",True
4,3 bedrooms in Irvine,"{'bedrooms': 3, 'city': 'Irvine'}","{'bedrooms': 3, 'city': 'Irvine'}",True


Total queries: 5
Correct queries: 5
Parsing accuracy: 100.00%
PASS: Accuracy meets the 90% requirement!


In [20]:

gold_queries = [
    # Bedroom queries
    ("3 bed", {"bedrooms": 3}),
    ("4 bedrooms", {"bedrooms": 4}),
    ("2br", {"bedrooms": 2}),
    ("3+ beds", {"bedrooms_min": 3}),
    ("at least 4 bedrooms", {"bedrooms_min": 4}),
    ("up to 5 bedrooms", {"bedrooms_max": 5}),
    ("max 2 beds", {"bedrooms_max": 2}),
    ("min 3 beds", {"bedrooms_min": 3}),

    # Bathroom queries
    ("2 bathrooms", {"bathrooms": 2}),
    ("3 bath", {"bathrooms": 3}),
    ("2ba", {"bathrooms": 2}),
    ("2+ bathrooms", {"bathrooms_min": 2}),
    ("at least 2 baths", {"bathrooms_min": 2}),
    ("3.5 baths", {"bathrooms": 3.5}),

    # Price queries
    ("under 700k", {"price_max": 700000}),
    ("below $800k", {"price_max": 800000}),
    ("under $1,000,000", {"price_max": 1000000}),
    ("under 1.2m", {"price_max": 1200000}),
    ("less than 600k", {"price_max": 600000}),
    ("up to 900k", {"price_max": 900000}),
    ("over 500k", {"price_min": 500000}),
    ("above $600,000", {"price_min": 600000}),
    ("at least 750k", {"price_min": 750000}),
    ("between 400k and 700k", {"price_min": 400000, "price_max": 700000}),
    ("from 500k to 900k", {"price_min": 500000, "price_max": 900000}),

    # City queries
    ("in Irvine", {"city": "Irvine"}),
    ("in Beverly Hills", {"city": "Beverly Hills"}),
    ("in Los Angeles", {"city": "Los Angeles"}),
    ("near Newport Beach", {"city": "Newport Beach"}),
    ("in Lathrop", {"city": "Lathrop"}),

    # Amenities
    ("with pool", {"amenities": ["pool"]}),
    ("with garage", {"amenities": ["garage"]}),
    ("with fireplace", {"amenities": ["fireplace"]}),
    ("with balcony", {"amenities": ["balcony"]}),
    ("with patio", {"amenities": ["patio"]}),
    ("with parking", {"amenities": ["parking"]}),
    ("with backyard", {"amenities": ["backyard"]}),

    # Negation
    ("without pool", {"exclude_amenities": ["pool"]}),
    ("no garage", {"exclude_amenities": ["garage"]}),
    ("exclude fireplace", {"exclude_amenities": ["fireplace"]}),

    # Combined queries
    ("3 bed under 700k", {"bedrooms": 3, "price_max": 700000}),
    ("4 bedrooms below 900k", {"bedrooms": 4, "price_max": 900000}),
    ("2+ beds under 500k", {"bedrooms_min": 2, "price_max": 500000}),
    ("3 bed in Irvine", {"bedrooms": 3, "city": "Irvine"}),
    ("4 bed in Lathrop under 1m", {"bedrooms": 4, "city": "Lathrop", "price_max": 1000000}),
    ("2 bed with pool", {"bedrooms": 2, "amenities": ["pool"]}),
    ("3 bed without garage", {"bedrooms": 3, "exclude_amenities": ["garage"]}),
    ("2 bath under 800k", {"bathrooms": 2, "price_max": 800000}),
    ("3 beds 2 baths", {"bedrooms": 3, "bathrooms": 2}),
    ("3 bed under 700k in Irvine", {"bedrooms": 3, "price_max": 700000, "city": "Irvine"})
]

print("Number of queries:", len(gold_queries))


## Step 2: Calculate accuracy automatically


import pandas as pd

results = []

for query, expected in gold_queries:
    predicted = parser.parse(query)

    correct = predicted == expected

    results.append({
        "Query": query,
        "Expected": expected,
        "Predicted": predicted,
        "Correct": correct
    })

df = pd.DataFrame(results)

accuracy = df["Correct"].mean() * 100

print("Total queries:", len(df))
print("Correct queries:", df["Correct"].sum())
print(f"Parsing Accuracy: {accuracy:.2f}%")

display(df)


Number of queries: 50
Total queries: 50
Correct queries: 49
Parsing Accuracy: 98.00%


,Query,Expected,Predicted,Correct
0,3 bed,{'bedrooms': 3},{'bedrooms': 3},True
1,4 bedrooms,{'bedrooms': 4},{'bedrooms': 4},True
2,2br,{'bedrooms': 2},{'bedrooms': 2},True
3,3+ beds,{'bedrooms_min': 3},{'bedrooms_min': 3},True
4,at least 4 bedrooms,{'bedrooms_min': 4},{'bedrooms_min': 4},True
5,up to 5 bedrooms,{'bedrooms_max': 5},{'bedrooms_max': 5},True
6,max 2 beds,{'bedrooms_max': 2},{'bedrooms_max': 2},True
7,min 3 beds,{'bedrooms_min': 3},{'bedrooms_min': 3},True
8,2 bathrooms,{'bathrooms': 2},{'bathrooms': 2},True
9,3 bath,{'bathrooms': 3},{'bathrooms': 3},True


In [21]:
import mysql.connector
import getpass
import json

password = getpass.getpass("MySQL root password: ")

conn = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=password,
    database="real_estate"
)

cursor = conn.cursor()

cursor.execute("""
    SELECT DISTINCT TRIM(L_City)
    FROM rets_property
    WHERE L_City IS NOT NULL
      AND TRIM(L_City) <> ''
    ORDER BY TRIM(L_City)
""")

cities = [row[0] for row in cursor.fetchall()]

cursor.close()
conn.close()

print("Total valid cities:", len(cities))
print("First 10 cities:", cities[:10])

Total valid cities: 979
First 10 cities: ['29 Palms', 'Acton', 'Adelanto', 'Agoura', 'Agoura Hills', 'Agua Dulce', 'Aguanga', 'Ahwahnee', 'Alameda', 'Alamo']
